# Part 3. Cleaning and merging
Author: **Dauirkhankyzy Balnur**

Input: files from Part 1 and Part 2.
Output:
- `data/fx_rates_clean.csv` - clean exchange rates (country, year, rate)
- `data/gdp_clean.csv` - clean GDP
- `data/merged_dataset.csv` - final table, one row per country
- `data/by_currency.csv` - one row per currency (for checking)

In [1]:
import re
import numpy as np
import pandas as pd

countries = pd.read_csv("data/wb_countries_raw.csv")
wb = pd.read_csv("data/wb_indicators_raw.csv")
gdp_raw = pd.read_csv("data/gdp_raw.csv", dtype=str)

print(countries.shape, wb.shape, gdp_raw.shape)

(296, 4) (5830, 5) (222, 4)


## 1. World Bank data
Remove groups (World, Euro area, ...) - in the country list they have region "Aggregates".

In [2]:
countries = countries[countries["region"] != "Aggregates"]
print("real countries:", len(countries))

wb = wb[wb["iso3"].isin(countries["iso3"])]
wb["year"] = wb["year"].astype(int)
print("rows left:", len(wb))
print("duplicates:", wb.duplicated(["iso3", "year", "indicator"]).sum())

real countries: 217
rows left: 4774
duplicates: 0


Make a table where rows are countries and columns are years.

In [3]:
fx = wb[wb["indicator"] == "exchange_rate"].pivot(index="iso3", columns="year", values="value")
infl = wb[wb["indicator"] == "inflation"].pivot(index="iso3", columns="year", values="value")
fx.head()

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
iso3,,,,,,,,,,,
ABW,1.790000,1.790000,1.790000,1.790000,1.790000,1.790000,1.790000,1.790000,1.790000,1.790000,1.790000
AFG,61.143462,67.866086,68.026904,72.083247,77.737949,76.813536,NaN,NaN,NaN,NaN,NaN
AGO,120.060702,163.656434,165.915951,252.855748,364.825805,578.258780,631.441956,460.567512,685.020238,869.846190,911.993038
ALB,125.961667,124.142500,119.100000,107.989167,109.850833,108.650000,103.520000,113.041667,100.645000,93.123333,NaN
AND,NaN,NaN,NaN,NaN,NaN,NaN,0.845377,0.950916,0.924710,0.924367,0.887144


## 2. Problems with exchange rates
1. Some countries have no data at all -> remove.
2. Some countries use the US dollar (rate is always 1) -> remove, the change vs USD is always 0.
3. Croatia changed kuna to euro in 2023, so the rate jumps from ~7 to ~0.9. We divide the old years by 7.5345 (official rate HRK/EUR).
4. Other countries where the rate fell more than 2 times in one year probably changed their currency -> remove.
5. We need 2015 and 2025 and at least 9 years of 11. Small gaps -> linear interpolation.

In [4]:
# 1
print("no data:", fx.isna().all(axis=1).sum())
fx = fx.dropna(how="all")

# 2
usd_countries = []
for iso in fx.index:
    values = fx.loc[iso].dropna()
    if iso == "USA" or (values.round(4) == 1).all():
        usd_countries.append(iso)
print("use USD:", usd_countries)
fx = fx.drop(usd_countries)

# 3
old_years = [y for y in fx.columns if y < 2023]
fx.loc["HRV", old_years] = fx.loc["HRV", old_years] / 7.5345

# 4
ratio = fx / fx.shift(1, axis=1)
changed = ratio[(ratio < 0.5).any(axis=1)].index.tolist()
print("currency change:", changed)
fx = fx.drop(changed)

# 5
good = fx[2015].notna() & fx[2025].notna() & (fx.notna().sum(axis=1) >= 9)
fx = fx[good]
print("countries left:", len(fx))
print("missing before:", fx.isna().sum().sum())
fx = fx.interpolate(axis=1)
print("missing after:", fx.isna().sum().sum())

no data: 5
use USD: ['ASM', 'BHS', 'BMU', 'ECU', 'FSM', 'GUM', 'MHL', 'MNP', 'PAN', 'PLW', 'PRI', 'SLV', 'TCA', 'TLS', 'USA', 'VIR']
currency change: ['ZWE']
countries left: 167
missing before: 1
missing after: 0


## 3. New columns for every country
- `value_change_pct` - how the value of the currency changed vs USD from 2015 to 2025: `(rate_2015 / rate_2025 - 1) * 100`. Negative = currency became weaker.
- `volatility_pct` - standard deviation of yearly changes
- `max_yearly_drop_pct` - worst year
- `avg_inflation_pct` - average inflation (only if there are at least 8 years)

In [5]:
# value of 1 unit of currency in dollars = 1 / rate
yearly_change = (1 / fx).pct_change(axis=1) * 100
yearly_change = yearly_change.drop(columns=2015)

features = pd.DataFrame()
features["rate_2015"] = fx[2015]
features["rate_2025"] = fx[2025]
features["value_change_pct"] = (fx[2015] / fx[2025] - 1) * 100
features["volatility_pct"] = yearly_change.std(axis=1)
features["max_yearly_drop_pct"] = yearly_change.min(axis=1)

infl_count = infl.notna().sum(axis=1)
features["avg_inflation_pct"] = infl.mean(axis=1)
features.loc[infl_count < 8, "avg_inflation_pct"] = np.nan

features = features.reset_index()
features = features.merge(countries, on="iso3", how="left")
features.sort_values("value_change_pct").head()

,iso3,rate_2015,rate_2025,value_change_pct,volatility_pct,max_yearly_drop_pct,avg_inflation_pct,name,region,income_level
3,ARG,9.233186,1240.851250,-99.255899,16.248679,-67.611272,NaN,Argentina,Latin America & Caribbean,Upper middle income
92,LBN,1507.500000,89500.000000,-98.315642,36.621196,-89.135625,63.716312,Lebanon,"Middle East, North Africa, Afghanistan & Pakistan",Lower middle income
156,TUR,2.720009,39.454813,-93.106016,10.313193,-46.519535,28.139172,Turkiye,Europe & Central Asia,Upper middle income
143,SUR,3.416667,37.260179,-90.830246,19.779844,-48.957094,34.197901,Suriname,Latin America & Caribbean,Upper middle income
118,NGA,192.440333,1518.380948,-87.325952,17.740313,-56.375304,17.696130,Nigeria,Sub-Saharan Africa,Lower middle income


## 4. Cleaning GDP

In [6]:
gdp = gdp_raw.copy()
gdp.columns = ["country", "gdp_imf", "gdp_wb", "gdp_un"]

def clean_name(name):
    # "China [ n 1 ]" -> "China"
    return re.sub(r"\[.*?\]", "", name).strip()

def clean_number(x):
    # "407,786 (2025)" -> 407786.0 ;  "— N/a" -> NaN
    x = re.sub(r"\(.*?\)", "", str(x))
    x = re.sub(r"\[.*?\]", "", x)
    x = x.replace(",", "").strip()
    if x.isdigit():
        return float(x)
    return np.nan

gdp["country"] = gdp["country"].apply(clean_name)
for col in ["gdp_imf", "gdp_wb", "gdp_un"]:
    gdp[col] = gdp[col].apply(clean_number)

gdp = gdp[gdp["country"] != "World"]
print("duplicates:", gdp["country"].duplicated().sum())
print(gdp.isna().sum())

duplicates: 0
country     0
gdp_imf    24
gdp_wb      7
gdp_un      9
dtype: int64


We take IMF GDP. If IMF is missing we take World Bank, then UN. Then convert millions to billions.

In [7]:
gdp["gdp_usd_bn"] = gdp["gdp_imf"].fillna(gdp["gdp_wb"]).fillna(gdp["gdp_un"]) / 1000

def which_source(row):
    if pd.notna(row["gdp_imf"]):
        return "IMF"
    if pd.notna(row["gdp_wb"]):
        return "World Bank"
    return "UN"

gdp["gdp_source"] = gdp.apply(which_source, axis=1)
gdp = gdp.dropna(subset=["gdp_usd_bn"]).sort_values("gdp_usd_bn", ascending=False)
gdp["gdp_rank"] = range(1, len(gdp) + 1)

print(len(gdp))
print(gdp["gdp_source"].value_counts())
gdp.head(10)

221
gdp_source
IMF           197
World Bank     18
UN              6
Name: count, dtype: int64


,country,gdp_imf,gdp_wb,gdp_un,gdp_usd_bn,gdp_source,gdp_rank
1,United States,32383920.0,30769700.0,29298000.0,32383.920,IMF,1
2,China,20851593.0,19498039.0,18743802.0,20851.593,IMF,2
3,Germany,5452858.0,5050923.0,4659929.0,5452.858,IMF,3
4,Japan,4379253.0,4435163.0,4026211.0,4379.253,IMF,4
5,United Kingdom,4264794.0,4002588.0,3685881.0,4264.794,IMF,5
6,India,4153191.0,3956067.0,3952244.0,4153.191,IMF,6
7,France,3596094.0,3366316.0,3160443.0,3596.094,IMF,7
8,Italy,2738164.0,2551557.0,2380825.0,2738.164,IMF,8
9,Russia,2656452.0,2561310.0,2173386.0,2656.452,IMF,9
10,Brazil,2635912.0,2279920.0,2185822.0,2635.912,IMF,10


## 5. Merging
We merge by country name, but World Bank and Wikipedia write some names differently.
First let's find which names do not match.

In [8]:
not_matched = sorted(set(countries["name"]) - set(gdp["country"]))
print(len(not_matched))
print(not_matched)

37
['Bahamas, The', 'Brunei Darussalam', 'Cabo Verde', 'Congo, Dem. Rep.', 'Congo, Rep.', "Cote d'Ivoire", 'Curacao', 'Czechia', 'Egypt, Arab Rep.', 'Gambia, The', 'Gibraltar', 'Hong Kong SAR, China', 'Iran, Islamic Rep.', "Korea, Dem. People's Rep.", 'Korea, Rep.', 'Kyrgyz Republic', 'Lao PDR', 'Macao SAR, China', 'Micronesia, Fed. Sts.', 'Naoero', 'Puerto Rico (US)', 'Russian Federation', 'Sao Tome and Principe', 'Sint Maarten (Dutch part)', 'Slovak Republic', 'Somalia, Fed. Rep.', 'St. Kitts and Nevis', 'St. Lucia', 'St. Martin (French part)', 'St. Vincent and the Grenadines', 'Syrian Arab Republic', 'Turkiye', 'Venezuela, RB', 'Viet Nam', 'Virgin Islands (U.S.)', 'West Bank and Gaza', 'Yemen, Rep.']


We fix them by hand with a dictionary (World Bank name -> Wikipedia name).

In [9]:
names = {
    "Bahamas, The": "Bahamas", "Brunei Darussalam": "Brunei", "Cabo Verde": "Cape Verde",
    "Congo, Dem. Rep.": "DR Congo", "Congo, Rep.": "Congo", "Cote d'Ivoire": "Ivory Coast",
    "Curacao": "Curaçao", "Czechia": "Czech Republic", "Egypt, Arab Rep.": "Egypt",
    "Gambia, The": "Gambia", "Hong Kong SAR, China": "Hong Kong", "Iran, Islamic Rep.": "Iran",
    "Korea, Dem. People's Rep.": "North Korea", "Korea, Rep.": "South Korea",
    "Kyrgyz Republic": "Kyrgyzstan", "Lao PDR": "Laos", "Macao SAR, China": "Macau",
    "Micronesia, Fed. Sts.": "Federated States of Micronesia", "Naoero": "Nauru",
    "Puerto Rico (US)": "Puerto Rico", "Russian Federation": "Russia",
    "Sao Tome and Principe": "São Tomé and Príncipe", "Sint Maarten (Dutch part)": "Sint Maarten",
    "Slovak Republic": "Slovakia", "Somalia, Fed. Rep.": "Somalia",
    "St. Kitts and Nevis": "Saint Kitts and Nevis", "St. Lucia": "Saint Lucia",
    "St. Martin (French part)": "Saint Martin",
    "St. Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "Syrian Arab Republic": "Syria", "Turkiye": "Turkey", "Venezuela, RB": "Venezuela",
    "Viet Nam": "Vietnam", "Virgin Islands (U.S.)": "U.S. Virgin Islands",
    "West Bank and Gaza": "Palestine", "Yemen, Rep.": "Yemen",
}

features["country"] = features["name"].replace(names)
print("still not found:", sorted(set(countries["name"].replace(names)) - set(gdp["country"])))

still not found: ['Gibraltar']


In [10]:
merged = features.merge(gdp[["country", "gdp_usd_bn", "gdp_rank", "gdp_source"]], on="country", how="inner")
merged["log10_gdp"] = np.log10(merged["gdp_usd_bn"])
merged = merged.sort_values("gdp_usd_bn", ascending=False)

merged = merged[["iso3", "country", "region", "income_level", "gdp_usd_bn", "log10_gdp", "gdp_rank",
                 "rate_2015", "rate_2025", "value_change_pct", "volatility_pct",
                 "max_yearly_drop_pct", "avg_inflation_pct", "gdp_source"]]

print("before merge:", len(features), "after merge:", len(merged))
print("duplicates:", merged["country"].duplicated().sum())
print(merged.isna().sum())
merged.head(10)

before merge: 167 after merge: 166
duplicates: 0
iso3                    0
country                 0
region                  0
income_level            0
gdp_usd_bn              0
log10_gdp               0
gdp_rank                0
rate_2015               0
rate_2025               0
value_change_pct        0
volatility_pct          0
max_yearly_drop_pct     0
avg_inflation_pct      20
gdp_source              0
dtype: int64


,iso3,country,region,income_level,gdp_usd_bn,log10_gdp,gdp_rank,rate_2015,rate_2025,value_change_pct,volatility_pct,max_yearly_drop_pct,avg_inflation_pct,gdp_source
30,CHN,China,East Asia & Pacific,Upper middle income,20851.593,4.319139,2,6.227489,7.189847,-13.384967,3.947792,-6.275725,1.444612,IMF
43,DEU,Germany,Europe & Central Asia,High income,5452.858,3.736624,3,0.901296,0.884969,1.844976,4.883245,-10.965355,2.377485,IMF
83,JPN,Japan,East Asia & Pacific,High income,4379.253,3.641400,4,121.044026,149.657927,-19.119536,7.340701,-16.535456,1.275331,IMF
58,GBR,United Kingdom,Europe & Central Asia,High income,4264.794,3.629898,5,0.654545,0.759474,-13.815942,6.246983,-11.623681,3.031244,IMF
76,IND,India,South Asia,Lower middle income,4153.191,3.618382,6,64.151944,87.158448,-26.396183,2.872647,-5.962099,4.755146,IMF
55,FRA,France,Europe & Central Asia,High income,3596.094,3.555831,7,0.901296,0.884969,1.844976,4.883245,-10.965355,1.761325,IMF
80,ITA,Italy,Europe & Central Asia,High income,2738.164,3.437459,8,0.901296,0.884969,1.844976,4.883245,-10.965355,1.908513,IMF
20,BRA,Brazil,Latin America & Caribbean,Upper middle income,2635.912,3.420931,10,3.326904,5.587286,-40.455809,9.308049,-23.485271,5.762233,IMF
26,CAN,Canada,North America,High income,2507.340,3.399213,11,1.278786,1.398106,-8.534366,3.291947,-3.663148,2.510571,IMF
6,AUS,Australia,East Asia & Pacific,High income,2123.963,3.327147,12,1.331090,1.551998,-14.233758,4.860968,-7.660604,2.745337,IMF


## 6. One row per currency
Many countries have the same currency (20 euro countries, CFA franc in Africa). Their rows are the same,
so they can change the correlations. We group countries with the same value change and volatility
(= same currency, or currencies fixed to the same thing, like all currencies fixed to USD)
and sum their GDP. Part 4 uses this to check the results.

In [11]:
merged["key"] = merged["value_change_pct"].round(3).astype(str) + "_" + merged["volatility_pct"].round(3).astype(str)

by_currency = merged.groupby("key").agg(
    country=("country", "first"),
    n_countries=("country", "count"),
    gdp_usd_bn=("gdp_usd_bn", "sum"),
    value_change_pct=("value_change_pct", "first"),
    volatility_pct=("volatility_pct", "first"),
    max_yearly_drop_pct=("max_yearly_drop_pct", "first"),
    avg_inflation_pct=("avg_inflation_pct", "mean"),
).reset_index(drop=True)
by_currency["log10_gdp"] = np.log10(by_currency["gdp_usd_bn"])
merged = merged.drop(columns="key")

print("countries:", len(merged), " currencies:", len(by_currency))
by_currency.sort_values("n_countries", ascending=False).head()

countries: 166  currencies: 99


,country,n_countries,gdp_usd_bn,value_change_pct,volatility_pct,max_yearly_drop_pct,avg_inflation_pct,log10_gdp
84,Germany,22,19200.477,1.844976,4.883245,-10.965355,2.612467,4.283312
74,Saudi Arabia,20,2583.661,0.000000,0.000000,0.000000,1.631144,3.412236
80,Ivory Coast,18,450.117,1.595250,4.917519,-11.098669,2.494399,2.653325
13,Australia,4,2124.625,-14.233758,4.860968,-7.660604,2.537026,3.327282
81,Denmark,3,511.152,1.602002,4.956063,-11.150676,1.814753,2.708550


## 7. Saving

In [12]:
fx_long = fx.stack().reset_index()
fx_long.columns = ["iso3", "year", "rate_per_usd"]

fx_long.to_csv("data/fx_rates_clean.csv", index=False)
gdp.to_csv("data/gdp_clean.csv", index=False)
merged.to_csv("data/merged_dataset.csv", index=False)
by_currency.to_csv("data/by_currency.csv", index=False)
print("saved")

saved
